# Option Pricer Analysis

Reads the results produced by the C++ pricer and visualises:
- Monte Carlo price convergence toward the Black-Scholes price (standard vs antithetic)
- The error vs numSims on a log-log scale, against the theoretical 1/sqrt(N) rate
- Computation time

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Read the CSV produced by the C++ pricer (path relative to python/).
# Columns: numSims, mcPrice, mcError, antiPrice, antiError, bsPrice, timeMs
df = pd.read_csv('../output/results.csv')
print(df)

In [ ]:
bs_price = df['bsPrice'].iloc[0]  # same for every row

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Monte Carlo vs Black-Scholes Option Pricing', fontsize=14)

# --- Plot 1: price convergence (standard vs antithetic) ---
ax1 = axes[0]
ax1.semilogx(df['numSims'], df['mcPrice'], 'o-', color='steelblue', label='Standard MC')
ax1.semilogx(df['numSims'], df['antiPrice'], 's-', color='seagreen', label='Antithetic MC')
ax1.axhline(y=bs_price, color='crimson', linestyle='--', label=f'BS price ({bs_price:.4f})')
ax1.set_xlabel('Number of simulations')
ax1.set_ylabel('Option price')
ax1.set_title('MC Convergence to BS Price')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: absolute error, standard vs antithetic, log-log ---
# A 1/sqrt(N) relationship appears as a straight line of slope -0.5 here.
ax2 = axes[1]
mc_err   = df['mcError'].abs()
anti_err = df['antiError'].abs()
N = df['numSims'].values

ax2.loglog(N, mc_err, 'o-', color='steelblue', label='|Standard error|')
ax2.loglog(N, anti_err, 's-', color='seagreen', label='|Antithetic error|')

# Theoretical 1/sqrt(N) reference, scaled to pass through the first standard point
scale = mc_err.iloc[0] * np.sqrt(N[0])
ax2.loglog(N, scale / np.sqrt(N), '--', color='crimson', label=r'$1/\sqrt{N}$ reference')

ax2.set_xlabel('Number of simulations')
ax2.set_ylabel('Absolute error')
ax2.set_title('Error vs Simulations (log-log)')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

# --- Plot 3: timing ---
ax3 = axes[2]
ax3.semilogx(df['numSims'], df['timeMs'], 'o-', color='steelblue', label='Standard MC time (ms)')
ax3.set_xlabel('Number of simulations')
ax3.set_ylabel('Time (ms)')
ax3.set_title('MC Computation Time')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to output/convergence.png')

In [ ]:
# Summary: how much variance reduction did antithetic variates buy us?
final = df.iloc[-1]
print(f'Black-Scholes price:           {bs_price:.6f}')
print(f'Standard MC  (N={int(final.numSims)}):   {final.mcPrice:.6f}  | error {final.mcError:+.6f}')
print(f'Antithetic MC (N={int(final.numSims)}):   {final.antiPrice:.6f}  | error {final.antiError:+.6f}')
ratio = abs(final.mcError) / abs(final.antiError)
print(f'\nAntithetic error is {ratio:.1f}x smaller at the largest N.')
print(f'Equivalent standard-MC speedup: ~{ratio**2:.0f}x more sims needed to match it.')